# LLM Insight Generation Pipeline

Take transcripts from the ingestion pipeline and pass them into an LLM to extract **claims**, **narratives**, and **trends**.

Additional notes --> things to implement:
- Write a Provider interface (LLMProvider.generateJSON(schema, text)).
-Force structured output (JSON) and validate it (Zod/Pydantic).

Implement 2 providers:
- OllamaProvider
- BedrockProvider
- Put provider choice behind env vars:
    - ``LLM_PROVIDER=ollama | bedrock``
    - ``LLM_MODEL=llama3 | anthropic.claude... | ...``

```
┌─────────────────────────────────────────┐
│    YouTube Transcripts (from earlier)   │
└──────────────────┬──────────────────────┘
                   │
                   ▼
        ┌──────────────────────┐
        │  LLM Insight Pipeline│
        └──────────┬───────────┘
                   │
         ┌─────────▼─────────┐
         │   Check Env Var   │
         │  LLM_PROVIDER?    │
         └─────────┬─────────┘
                   │
        ┌──────────┴──────────┐
        │                     │
        ▼                     ▼
   ┌─────────────┐       ┌──────────────┐
   │   Ollama    │       │   Bedrock    │
   │  (Local)    │       │  (AWS Cloud) │
   └──────┬──────┘       └──────┬───────┘
          │                     │
          └──────────┬──────────┘
                     │
                     ▼
        ┌──────────────────────┐
        │   Extract & Format   │
        │ • Claims             │
        │ • Narratives         │
        │ • Trends             │
        │ (As structured JSON) │
        └──────────────────────┘
```

In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Protocol
from abc import ABC, abstractmethod

# Using Dataclasses --> less boilerplate bs
@dataclass
class TranscriptRecord:
    """
    This data model represents one individual video transcript, with the following attributes:
    - video_id: unique identifier for each video, from YouTube's API
    - cleaned_txt_path: path to cleaned .txt
    - raw_json_path: raw JSON transcript (includes timestamps, segments for source-tracking)
    """

    video_id: str
    cleaned_txt_path: Path
    raw_json_path: Optional[Path] = None

@dataclass
class GeneratedInsights:
    """
    This data model houses. LLM-generated outputs + video metadata for storage
    """
    
    video_id: str
    claims: List[Dict[str, Any]]
    narratives: List[Dict[str, Any]]
    model: str
    provider: str
    source_cleaned_txt: str
    source_raw_json: Optional[str]
    chunk_count: int



class LLMProvider(ABC):
    """
    Base interface structure --> Any LLM provider in this pipeline must use this class. 
    - Allows for faster migration/transition from one LLM to another
        --> ie. Ollama (local) to AWS Bedrock/Kendra

    Each provider must define 2 attributes:
    - provider = (OpenAI, Anthropic, Google, etc.)
    - model = specific LLM configuration we're working with
    """
    def __init__(self, provider: str, model:str):
        self.provider = provider
        self.model = model

    @abstractmethod
    def generate_response(self, *, system: str, user_prompt: str) -> str:
        """
        Takes a system message & user prompt for input and returns raw JSON formatted responses.
        - Interface doesn't fully outline function behavior becuase response generation varies from provider to provider.
            --> HTTP Call?
            --> boto3 SDK Call?
        """
        pass

## 1. Load & Build Transcripts

Read cleaned transcripts from storage. Support single or batch. Keep source metadata for traceability.

In [ ]:
import os, sys
from pathlib import Path

TRANSCRIPT_ROOT_DIR = "data/transcripts"

def load_cleaned_transcripts(fp: str) -> str:
    file_path = Path(fp)
    
    for file in file_path:
        if file.suffix == ".txt":
            text = file.read_text(encoding="utf-8")
        else:
            sys.exit("File is not formatted as a .txt file")

## 2. Prepare for Analysis

Format transcript for the model. Handle length limits (chunk, truncate, or summarize). Handle edge cases.

In [ ]:
import json
from typing import Any, Dict, List

def chunk_text(text: str, max_chars: int = 12000) -> List[str]:
    """
    Splits long transcript text into chunks to fit model context limits.
    Uses paragraph boundaries when possible.
    """
    # PSEUDOCODE:
    # 1. Split text on paragraph boundaries (e.g. double newline)
    # 2. Iterate over paragraphs, accumulating into a buffer
    # 3. When adding next paragraph would exceed max_chars, flush buffer to chunks, start new buffer
    # 4. Append final buffer to chunks
    # 5. Return list of chunk strings
    pass

def validate_json_output(text: str) -> Dict[str, Any]:
    """
    Parses and sanity-checks the model output as JSON.
    """
    # PSEUDOCODE:
    # 1. Parse text as JSON
    # 2. Ensure top-level is a dict
    # 3. Ensure required keys exist (e.g. "claims", "narratives")
    # 4. Return parsed dict, or raise on validation failure
    pass

## 3. Define Output Structure

Specify what **claims**, **narratives**, and **trends** look like. Use this to instruct the model and parse responses.

In [ ]:
SYSTEM = "You extract factual claims and high-level narratives. Return ONLY valid JSON."
SCHEMA_HINT = """
{
  "claims": [{"text": "string", "speaker": "string|null", "confidence": 0.0, "evidence": ["string"]}],
  "narratives": [{"theme": "string", "summary": "string", "evidence": ["string"]}]
}
""".strip()

def build_user_prompt(transcript_chunk: str) -> str:
    """
    Builds the user prompt for the LLM, including the JSON shape we expect.
    """
    # PSEUDOCODE:
    # 1. Combine SCHEMA_HINT + transcript chunk into a single prompt string
    # 2. Instruct model to return ONLY valid JSON matching the schema
    # 3. Return the formatted prompt
    pass

def extract_insights_for_record(record: TranscriptRecord, provider: LLMProvider, *, max_chars: int = 12000, retries: int = 2) -> GeneratedInsights:
    """
    Loads the cleaned transcript, chunks it, runs the provider on each chunk,
    merges results, and returns a single GeneratedInsights.
    """
    # PSEUDOCODE:
    # 1. Load transcript text from record.cleaned_txt_path (use load_cleaned_transcripts)
    # 2. Chunk text via chunk_text()
    # 3. For each chunk:
    #    a. Build user prompt via build_user_prompt(chunk)
    #    b. Call provider.generate_response(system=SYSTEM, user_prompt=...) with retries
    #    c. Parse response via validate_json_output()
    #    d. Extend all_claims and all_narratives from parsed result
    # 4. Deduplicate claims (e.g. by claim text)
    # 5. Return GeneratedInsights(video_id, claims, narratives, model, provider, source paths, chunk_count)
    pass

## 4. Run Model

Send transcript + instructions to the model. Retrieve the response. Handle failures and limits.

Separate implementation for:
- Ollama
- AWS Bedrock

In [ ]:
class OllamaProvider(LLMProvider):
    """
    Calls local Ollama using an OpenAI-compatible endpoint.
    """
    name = "ollama"

    def __init__(self, model: str = "llama3", base_url: str = "http://localhost:11434/v1"):
        super().__init__(provider="ollama", model=model)
        self.base_url = base_url.rstrip("/")

    def generate_response(self, *, system: str, user_prompt: str) -> str:
        # PSEUDOCODE:
        # 1. Build request payload: model, messages (system + user), temperature
        # 2. POST to base_url/chat/completions (OpenAI-compatible API)
        # 3. Raise on HTTP error
        # 4. Extract content from response (choices[0].message.content)
        # 5. Return raw string (expected to be JSON)
        pass

In [ ]:
class BedrockProvider(LLMProvider):
    """
    Calls Amazon Bedrock using the Converse API.
    """
    name = "bedrock"

    def __init__(self, model: str, region: str = "us-east-1"):
        super().__init__(provider="bedrock", model=model)
        self.region = region
        # PSEUDOCODE: init bedrock-runtime client for region

    def generate_response(self, *, system: str, user_prompt: str) -> str:
        # PSEUDOCODE:
        # 1. Call converse API: modelId, system message, user message, inference config (temp, maxTokens)
        # 2. Extract text from response (output.message.content[0].text)
        # 3. Return raw string (expected to be JSON)
        pass

## 5. Extract & Validate Insights

Parse the model response into structured insights. Validate against the defined structure. Link back to source.

## 6. Persist Insights

Write insights to storage. Use a consistent format. Handle re-runs (idempotency or versioning).

## Flow

Load → Prepare → Define structure → Run model → Extract & validate → Persist

In [ ]:
def build_transcript_records(cleaned_dir: str, raw_json_dir: str | None) -> list[TranscriptRecord]:
    """
    Scans cleaned_dir (and optionally raw_json_dir) and builds TranscriptRecord for each transcript.
    """
    # PSEUDOCODE:
    # 1. List .txt files in cleaned_dir
    # 2. For each file: derive video_id (e.g. from filename), resolve cleaned_txt_path
    # 3. If raw_json_dir: try to find matching raw JSON by video_id
    # 4. Yield/return list of TranscriptRecord
    pass

def run_batch(cleaned_dir: str, raw_json_dir: str | None, provider: LLMProvider):
    """
    Builds records from your folder, runs extraction per transcript,
    and prints a quick summary per file.
    """
    # PSEUDOCODE:
    # 1. records = build_transcript_records(cleaned_dir, raw_json_dir)
    # 2. For each record: call extract_insights_for_record(record, provider)
    # 3. On success: print summary (video_id, claim count, narrative count, chunks)
    # 4. On failure: print error, continue to next
    # 5. (Optional) Persist each GeneratedInsights to storage

    pass